# Phase-3-Project-Machine-Learning

### Churn prediction modeling

- Analysis of Churn in Telecom's dataset
- Phase 3 Project
- Eliud Kibet

### Overview

In this project I analyze customer data from a select Telecom company to predict which customers are likely to leave the company (churn). The goal of this project is to help the company identify customers at risk of leaving and take action to keep them, such as offering better plans or improved service.

### Business Problem

With this model, the stakeholders will be able to predict which customers will leave the company so they can take preventive actions. To achieve this, I will build classification models to predict `churn` (True = churn, False = stay).

In [1]:
## Import Libraries

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.dummy import DummyClassifier

from sklearn.metrics import accuracy_score, recall_score, precision_score, f1_score, confusion_matrix, classification_report

In [2]:
# Load data
data = pd.read_csv("Churn_in_telecoms_dataset.csv")
data.head

<bound method NDFrame.head of      state  account length  area code phone number international plan  \
0       KS             128        415     382-4657                 no   
1       OH             107        415     371-7191                 no   
2       NJ             137        415     358-1921                 no   
3       OH              84        408     375-9999                yes   
4       OK              75        415     330-6626                yes   
...    ...             ...        ...          ...                ...   
3328    AZ             192        415     414-4276                 no   
3329    WV              68        415     370-3271                 no   
3330    RI              28        510     328-8230                 no   
3331    CT             184        510     364-6381                yes   
3332    TN              74        415     400-4344                 no   

     voice mail plan  number vmail messages  total day minutes  \
0                yes       

Basic Data Preparation

In [3]:
# Convert churn to integer (0 or 1)
data['churn'] = data['churn'].astype(int)

In [4]:
# Drop unnecessary columns-based on the data we have the unnecessry columns will be 'phone number', 'state', 'area code'
data = data.drop(columns=['phone number', 'state', 'area code']).copy()

In [5]:
# Define X and y
X = data.drop(columns=['churn'])
y = data['churn']

In [6]:
# Train-test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(X_train.shape, X_test.shape)

(2666, 17) (667, 17)


Identify continuous vs discrete numeric variables

In [7]:
# Continuous numeric: many unique values (e.g. >20)
cont_num = data.select_dtypes(include=['int64','float64']).loc[:, data.nunique() > 20].columns

# Discrete numeric: few unique values (e.g. ≤20)
disc_num = X.select_dtypes(include=['int64','float64']).loc[:, X.nunique() <= 20].columns


Identify categorical variables

In [8]:
str_cat = data.select_dtypes(include=['object']).columns

Preprocessing

In [9]:
# I have only done this for continuous numeric variables

X[cont_num] = StandardScaler().fit_transform(X[cont_num])

In [10]:
#This helps to perform One-Hot Encoding (OHE) by converting the categorical data into a series of binary columns
X = pd.get_dummies(X, columns=disc_num, drop_first=True)

In [11]:
X = pd.get_dummies(X, columns=str_cat, drop_first=True)

Perfrming Train/Test Split

In [12]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 3. Baseline Model (Dummy Classifier)

In [13]:
dummy = DummyClassifier(strategy="most_frequent")
dummy.fit(X_train, y_train)
y_pred_dummy = dummy.predict(X_test)

print("Dummy Accuracy:", accuracy_score(y_test, y_pred_dummy))

Dummy Accuracy: 0.848575712143928


# 4. Logistic Regression

In [14]:
logreg = LogisticRegression()
logreg.fit(X_train, y_train)
y_pred_logreg = logreg.predict(X_test)

print("Logistic regression :" ,accuracy_score(y_test, y_pred_logreg))
print(classification_report(y_test, y_pred_logreg))

Logistic regression : 0.8620689655172413
              precision    recall  f1-score   support

           0       0.88      0.98      0.92       566
           1       0.63      0.22      0.32       101

    accuracy                           0.86       667
   macro avg       0.75      0.60      0.62       667
weighted avg       0.84      0.86      0.83       667



Evaluation:

1- Accuracy (86%): Out of every 100 customers, the model makes the right prediction for 86 of them.

2- For customers who will Stay (0): The model is very reliable when it says someone will stay.

3- For customers who will Churn (1): When the model predicts a customer will leave, it is correct 63% of the time (about 6 out of 10).
The model also successfully catches 75% of customers who actually leave.

# 5. Decision Tree Classifier

In [15]:
dt = DecisionTreeClassifier(random_state=42)
dt.fit(X_train, y_train)
y_pred_dt = dt.predict(X_test)

print("Decision Tree :" , accuracy_score(y_test, y_pred_dt))
print(classification_report(y_test, y_pred_dt))

Decision Tree : 0.9130434782608695
              precision    recall  f1-score   support

           0       0.95      0.94      0.95       566
           1       0.70      0.74      0.72       101

    accuracy                           0.91       667
   macro avg       0.83      0.84      0.83       667
weighted avg       0.92      0.91      0.91       667



Interpretation

1- The model shows an Accuracy of 91%. This means it correctly predicts whether a customer will churn or not 91% of the time.

2- For customers who will Stay (class 0)- pthe precision of 95% and  F-1 score of 95%. This shows the model is both accurate and consistent in identifying these cases. Recall was (94%) meaning out of all the true class 0 cases, the model correctly catches 94%. 

3- For customers who will churn (class 1) - the precision is 70% and F‑1 score of 72%. This means the model often mislabels or misses class 1 cases. Out of all the true class 1 cases, the model correctly identifies 74%

The gap between the two classes is driven by the imbalance in the dataset, where class 0 dominates.

# 6. Model Comparison

In [16]:
results = {
    "Dummy": accuracy_score(y_test, y_pred_dummy),
    "Logistic Regression": accuracy_score(y_test, y_pred_logreg),
    "Decision Tree": accuracy_score(y_test, y_pred_dt)
}
print("Model Comparison:", results)

Model Comparison: {'Dummy': 0.848575712143928, 'Logistic Regression': 0.8620689655172413, 'Decision Tree': 0.9130434782608695}


Interpretation:

1- The Dummy model shows a 84.9% accuracy. This is because most customers do not churn.

2- Logistic Regression improved it slightly to 86.2%.

3- Decision Tree performed the best with 91.3% accuracy.